In [75]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import cartopy.feature as cfeature
import datetime as dt
from IPython.display import Markdown as md
import os, re
import datetime as dt


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneOut

In [56]:
# import and convert MDB mean and total rainfall to panda series
mdb_mean = xr.open_dataset('data/mdb_mean_rainfall.nc')
mdb_total = xr.open_dataset('data/mdb_total_rainfall.nc')

mdb_mean_series = mdb_mean['precip'].to_series()
mdb_total_series = mdb_total['precip'].to_series()

In [57]:
# import and convert hadisst roni and oni to panda series
oni = xr.open_dataset('data/oni_hadisst.nc')
roni = xr.open_dataset('data/roni_hadisst.nc')

oni_series = oni['sst'].to_series() # 3 month rolling average (centered, e.g. JJA - July)
roni_series = roni['sst'].to_series()

In [58]:
roni_series.index.month.values

array([1, 2, 3, ..., 5, 6, 7], dtype=int32)

In [59]:
# Indices values are indexed as centre month. 
# But want to change index as DJF - Feb, so less confusion in calculating/understanding lead time. 
# If JJA: 8, then when predicting SON, it will be 1-month lead forecast (bc August is in between)
# then when AMJ (6) predicts SON, it will be a 3-month lead forecast (6,7,8 is in between)

# extract predictor dataset (oni & roni)  

def shift_index_one_month(data):
    data = data.copy()
    data.index = data.index + pd.DateOffset(months=1)
    return data

oni_data = shift_index_one_month(oni_series)
roni_data = shift_index_one_month(roni_series)
roni_data

time
1870-02-16 11:59:59.505615234         NaN
1870-03-14 23:59:59.340820312   -0.612827
1870-04-16 11:59:59.340820312   -0.505451
1870-05-15 23:59:59.340820312   -0.541640
1870-06-16 12:00:00.000000000   -0.583550
                                   ...   
2026-04-16 12:00:00.000000000   -0.377770
2026-05-16 00:00:00.000000000    0.004068
2026-06-16 12:00:00.000000000    0.485978
2026-07-16 00:00:00.000000000    0.976325
2026-08-16 12:00:00.000000000         NaN
Name: sst, Length: 1879, dtype: float32

In [60]:
# set all data to 1900 onwards (most available data) 

dt1900 = dt.datetime(1900,2,16)

roni_data = roni_data[dt1900:].copy()
oni_data = oni_data[dt1900:].copy()
mdb_mean_data = mdb_mean_series[dt1900:].copy()
mdb_total_data = mdb_total_series[dt1900:].copy()


In [61]:
# 1 month lead predictors: 3m mean june july august
roni_jja = roni_data[roni_data.index.month == 9].copy()
oni_jja = oni_data[oni_data.index.month == 9].copy()

# 3 month lead predictors: 3m mean april may june
roni_amj = roni_data[roni_data.index.month == 6].copy()
oni_amj = oni_data[oni_data.index.month == 6].copy()


# SON rainfall OVER TOTAL MDB !!! 
# to see MEAN MDB, change 'mdb_total_data' to 'mdb_mean_data'
mdb_data = mdb_mean_data

tp_son = (
    mdb_data[mdb_data.index.month.isin([9, 10, 11])]
    .groupby(mdb_data[mdb_data.index.month.isin([9, 10, 11])].index.year)
    .sum()
)

mp_son = (
    mdb_data[mdb_data.index.month.isin([9, 10, 11])]
    .groupby(mdb_data[mdb_data.index.month.isin([9, 10, 11])].index.year)
    .mean()
)
        

In [54]:
## Check we have useful data
# hadisst roni oni_jja only available up to 2025 bc 2026 data was up to july (mjj), update now ? 
for df,name in zip([roni_jja,oni_jja,roni_amj,oni_amj,tp_son, mp_son],
                   ['roni_jja','oni_jja','roni_amj','oni_amj','tp','mp']):
    print("===  ",name,"  ===")
    print("  data available from ",df.index[0], " to ", df.index[-1])
    
    print('  last five entries:')
    print(df[-5:])
    print() # breakline

===   roni_jja   ===
  data available from  1900-09-16 12:00:00  to  2025-09-16 12:00:00
  last five entries:
time
2021-09-16 12:00:00   -0.573886
2022-09-16 12:00:00   -0.976900
2023-09-16 12:00:00    1.046599
2024-09-16 12:00:00   -0.458692
2025-09-16 12:00:00   -0.425110
Name: sst, dtype: float32

===   oni_jja   ===
  data available from  1900-09-16 12:00:00  to  2025-09-16 12:00:00
  last five entries:
time
2021-09-16 12:00:00   -0.415976
2022-09-16 12:00:00   -0.911636
2023-09-16 12:00:00    1.422412
2024-09-16 12:00:00   -0.025739
2025-09-16 12:00:00   -0.205636
Name: sst, dtype: float32

===   roni_amj   ===
  data available from  1900-06-16 12:00:00  to  2026-06-16 12:00:00
  last five entries:
time
2022-06-16 12:00:00   -1.176597
2023-06-16 12:00:00    0.136041
2024-06-16 12:00:00   -0.090947
2025-06-16 12:00:00   -0.412333
2026-06-16 12:00:00    0.485978
Name: sst, dtype: float32

===   oni_amj   ===
  data available from  1900-06-16 12:00:00  to  2026-06-16 12:00:00
  last 

In [72]:
def combo_method_SKL_Dine(X1,X2,Y,
                     prediction_year=None,
                     ):
    """
        ARGS:
            X1,X2: input pandas Series for LDA1 and LDA2 
                these are expected to be nino3.4 anom 3m mean and relative nino3.4 anom 3m mean
            Y: pandas Series with mean SON rainfall over MDB and datetime index
            prediction_year=None : if not set assumed to be final year
            
        Returns:
            prediction_probabilities: numpy array[n_predictions,n_classes]
                row = predicted probabilities for each year in the 
                prediction range being below the median (class0)
                around median (class1)
                or above the median (class2)
    """
    ## Training and prediction ranges as list of indices
    if prediction_year is None:
        prediction_year = X1.index.year.values[-1]
    training_range = range(1900,prediction_year)
    i_tra = X1.index.year.isin(training_range)
    # final year of Y is being predicted, so array has 1 less length
    i_tra_Y = Y.index.isin(training_range)
    i_ver = X1.index.year.isin([prediction_year])
    # i_ver looks like [False,False,...,True] 
    # true where year == training range years
    
    ## X and y from data frame
    X_roni= X1.copy()[i_tra]
    X_roni_ver = X1.copy()[i_ver]
    X_oni = X2.copy()[i_tra]
    X_oni_ver = X2.copy()[i_ver]
    
    
    # y is based on median: 0 for less, 1 for around, 2 for over 
    y0 = Y[i_tra_Y]
    q33, q67 = np.quantile(y0, [1/3, 2/3])
    y = np.ones(len(y0),dtype=int)
    y[y0.values < q33] = 0
    # y[(y0.values >= q33) & (y0.values <= q67)] = 1
    y[y0.values > q67] = 2
    
    ## Initialise LDA models
    LDA_roni = LinearDiscriminantAnalysis(solver='svd',priors=[1/3,1/3,1/3])
    LDA_oni = LinearDiscriminantAnalysis(solver='svd',priors=[1/3,1/3,1/3])
    
    ## fit models using training data
    LDA_roni.fit(X_roni.values.reshape(-1, 1), y)
    LDA_oni.fit(X_oni.values.reshape(-1, 1), y)
    
    ## Predict probability of class0,class1,class2 for prediction input sets
    # reshape required as we are only using 1 input column
    prob_roni = LDA_roni.predict_proba(X_roni_ver.values.reshape(-1,1))
    prob_oni = LDA_oni.predict_proba(X_oni_ver.values.reshape(-1,1))

    # probabilities are given as array[n_predictions,n_classes]
    # so 3 columns, one for class 0, second for class 1, third for class 2
    
    return prob_roni, prob_oni

In [73]:
mp_son

time
1900    18.815821
1901    26.543703
1902    25.139946
1903    59.328960
1904    32.615124
          ...    
2021    67.605301
2022    99.020844
2023    35.502544
2024    39.752598
2025    29.531301
Name: precip, Length: 126, dtype: float32

In [74]:
# run prediction

PREDICTION_YEAR=2025 # 2025 for jja, 2026 for amj

# outputs for each target will be stored as a list
output_row        = []
# list will have several named entries and be put into a table
output_columns = [
    'P_roni(t1)','P_roni(t2)','P_roni(t3)',
    'P_oni(t1)','P_oni(t2)','P_oni(t3)',
    'rainfall mean','rainfall SD','rainfall median'
    ]

Y=mp_son
proba_roni, proba_oni = combo_method_SKL_Dine(
        X1=roni_jja,
        X2=oni_jja,
        Y=Y,
        prediction_year=PREDICTION_YEAR,
    )

    
    # bom probability is the mean from two LDAs
    # probas = (proba_nino+proba_soi)/2.0
    ## Tercile probability (?)
    #t1,t2,t3 = get_terciles(probas)
    ## Mean and std of SON mean rainfall over MBD over 1900-2025 , include 2025 or not?
mean = Y.mean()
std = Y.std()
median = Y.median()
    
## Add row to list
output_row.append([proba_roni[0,0],proba_roni[0,1],proba_roni[0,2],
                   proba_oni[0,0],proba_oni[0,1],proba_oni[0,2],
                   mean,std,median])
    
## Make list of rows into dataframe for easy printing...
df_stats=pd.DataFrame(output_row,columns=output_columns)

# Check output
md("# Prediction for %d"%(PREDICTION_YEAR))
print(df_stats)

   P_roni(t1)  P_roni(t2)  P_roni(t3)  P_oni(t1)  P_oni(t2)  P_oni(t3)  \
0    0.163617    0.317041    0.519342   0.306372    0.36005   0.333578   

   rainfall mean  rainfall SD  rainfall median  
0      38.985409    16.732161        35.730614  


In [ ]:
# Define BSS & LEPS & Hit/miss (contingency table, accuracy, bias)

def brier_skill_score(obs,proba, 
    assumed_BS_ref_prob=1/3, show_calculations=False,
    ):
    """ 
        Brier Skill is essentially mean square error:
            (prediction proba - [0 if true, 1 if false])**2
        BSS = 1 - BS/BS_ref
        BS_ref = score of reference method we are trying to beat
            in this case BS_ref is an assumed 33.333% for each tercile
        
        BSS = 1 the forecast has perfect skill compared to climatology;
        BSS = 0 the forecast has no skill compared to climatology);
        BSS = a negative value the forecast is less accurate than climatology.
        
        ARGUMENTS:
            proba is the probability distribution
            obs is list of class 0, 1 or class 2 occurrences
            assumed_... is reference climatological model estimate(s) of class 0,1,2
                this could be updated to (for example) the number of class 1 outcomes / total number of predictions
                currently is set to 1/3% (expect 1/3 to be greater than median, 1/3 to be =, 1/3 to be below)
    """
    # if proba is [n rows, 3 columns] then it is probably [prob_below;prob_middle;prob_above]
    # fix it with warning
    if len(np.shape(proba)) == 2 and np.shape(proba)[1]==3: # if 2D array (n_years,tercile) and 3 columns
        print("WARNING: brier_skill_score proba is assuming the first column to be probability of class 0")
        t1 = proba[:,0]
        t2 = proba[:,1]
        t3 = proba[:,2]
        

    # remove any missing values
    df = pd.DataFrame({'obs':obs,'t1':t1, 't2':t2, 't3':t3})
    df = df.dropna()
    
    # Brier Score: 0 if 100% prediction is correct, 1 if 100% prediction is wrong
    df['BS'] = (
        (df['t1' - (df['obs']==0).astype(int)) **2 + # If t1 is assigned 1.00 at lower tercile (obs), then 1-1 = 0 => perfect prediction  
        (df['t2' - (df['obs']==1).astype(int)) **2 + 
        ((df['t3' - (df['obs']==2).astype(int)) **2)
    )/3
    df['BS_ref'] = (
        (assumed_BS_ref_prob - (df['obs']==0).astype(int)) **2 + 
        (assumed_BS_ref_prob - (df['obs']==1).astype(int)) **2 + 
        (assumed_BS_ref_prob - (df['obs']==2).astype(int)) **2
    )/3
    df['BSS'] = 1 - df['BS']/df['BS_ref']
    if show_calculations:
        print(df)
    # return mean of BSS
    BSS = df['BSS'].mean()

    return BSS

def calculate_classed_leps_skill(y_obs, probabilities):
    """
 
    ARGUMENTS:
        y_obs: array[n] of classes observed
        probabilities: array[n,3] of probabilities to be in class (0 to 2) for n observations
            NB: predictions can be inferred from this, but aren't necessary here

    Added by DY: 
    Tercile (three-category) LEPS described in Fawcett et al., 2005. 
    Suppose the probability of a tercile 1 outcome at a given point is given as p1, 
    with the corresponding probabilities for terciles 2 and 3 being p2 and p3 respectively. 
    Obviously, p1 + p2 + p3 = 1.
    If the outcome is in tercile 1 then the forecast is given the LEPS score s = 8/27 p1 – 1/27 p2 – 7/27 p3, 
    if the outcome is  in tercile 2 then the forecast is given the LEPS score s = –1/27 p1 + 2/27 p2 – 1/27 p3, 
    and if the outcome is  in tercile 3 then the forecast is given the LEPS score s = –7/27 p1 – 1/27 p2 + 8/27 p3. 
    These scores range  from –7/27 to +8/27, but can be multiplied by 27/8 to give a scaled LEPS score which ranges from –7/8 to +1. 
    For a sequence {s1,...,sn} of these LEPS scores, calculate two additional sequences, 
    {u1,...,un} which is the sequence of maximum possible LEPS scores given the observed outcomes, and
    {l1,...,ln} which is the sequence of minimum possible LEPS scores given the observed outcomes.
    Obviously this implies that li ≤ si ≤ ui for i = 1,...,n.
    If the outcome for the ith forecast is a tercile 1 or tercile 3 outcome, then ui = 8/27 and li =  –7/27, 
    whereas if the outcome for the ith forecast is a tercile 2 outcome, then ui = 2/27 and li = –1/27.

    The LEPS skill-score for the sequence of forecasts is calculated as
    LEPS skill = (sum (i from 1 to n) s_i) / (sum (i from 1 to n) u_i)) if the mean LEPS score is non-negative, and as 
    LEPS skill = (sum (i from 1 to n) s_i) / (sum (i from 1 to n) l_i)) otherwise. 
    
    The LEPS skill-score can range from –1 to +1.
    The LEPS score base rate for individual forecasts is zero in both the two-category and three-category cases, 
    this being the score awarded to a climatological forecast (p1 = p2 = 1/2 in the two-category case and p1 =  p2 = p3 = 1/3 in the three-category case). 
    The base  rate for the LEPS skill-score is also zero in both cases.

    """

    # According to Fawcett et al. (2005):
    # tercile scoring matrix
    score_matrix = np.array([
    [ 8, -1, -7],   
    [-1,  2, -1],   
    [-7, -1,  8]    
    ]) / 27

    # LEPS score
    s = np.zeros(len(y_obs))

    for i in range(len(y_obs)):
        observed_class = y_obs[i]
        probabilities_i = probabilities[i] 

    s[i] = np.sum(
        probabilities_i * score_matrix[observed_class]
    )
    
    print (s.shape)
    
    # maximum and minimum possible LPES scores
    u = np.where(
        (y_obs == 0) | (y_obs == 2), # if obs is t1 or t3 
        8/27, # highest for t1, t3
        2/27 # otherwise, highest for t2
    )

    l = np.where(
        (y_obs == 0) | (y_obs == 2), # if obs is t1 or t3
        -7/27, # lowest for t1, t3
        -1/27 # otherwise, lowest for t2
    )
    
    print (u.shape)
    print (l.shape)

   # LEPS skill score
   if np.mean(s) >= 0:
       leps_skill = np.sum(s) / np.sum(u)
   else:
       leps_skill = np.sum(s) / np.sum(l)

    return leps_skill



def get_classification_scores(y_prob,y_obs):
    """
    ARGS:
        y_prob: probabilities of class numpy array [n,2]: [[t1, t2, t3],...]
        y_class: true class: [0,1,2,1,...]
    """
    scores={}

    # Predicted tercile = class with highest probability
    y_pred = np.argmax(y_prob, axis=1) #np.argmax returns index-class (0,1,2) of highest probability

    # Build contingency table
    # yrows = predictions, xcolumns = observations
    contingency = np.zeros((3, 3), dtype=int)
    for pred, obs in zip(y_pred, y_obs):
        contingency[pred, obs] += 1 # adds 1 to according (row, column) 

    scores['contingency'] = contingency

    # Hit and misses for each tercile (diagonal values in contingency table)
    scores['hit_t1'] = contingency[0, 0]
    scores['hit_t2'] = contingency[1, 1]
    scores['hit_t3'] = contingency[2, 2]

    scores['miss_t1'] = contingency[1, 0] + contingency[2, 0]
    scores['miss_t2'] = contingency[0, 1] + contingency[2, 1]
    scores['miss_t3'] = contingency[0, 2] + contingency[1, 2]

    # Overall accuracy 0 to 1
    # (correct predictions) / (number of predictions) 
    scores['accuracy'] = np.mean(y_pred == y_obs)

    # Save observations and probabilities
    scores['class'] = y_obs
    scores['prob_t1'] = y_prob[:, 0]
    scores['prob_t2'] = y_prob[:, 1]
    scores['prob_t3'] = y_prob[:, 2]

    ## bias = (class predictions) / (class occurrences)
    # bias = 1 predicted frequency matches observed frequency
    # bias < 1 underforecast the tercile
    # bias > 1 overforecast the tercile 
    # bias is 0 to infinity, ignore divide by zero warning

    with np.errstate(divide='ignore', invalid='ignore'): # ignore division by 0, invalid division or operation
        scores['bias_t1'] = np.sum(y_pred == 0) / np.sum(y_obs == 0) # tercile divided by 
        scores['bias_t2'] = np.sum(y_pred == 1) / np.sum(y_obs == 1)
        scores['bias_t3'] = np.sum(y_pred == 2) / np.sum(y_obs == 2)

    
    LEPS_test=calculate_classed_leps_skill(y_obs,y_prob)
    scores['LEPS skill']=(LEPS_test)
    scores['BSS'] = brier_skill_score(y_obs,y_prob)

    
    return scores

In [ ]:
# From 7-9
# Define validatio/training methods
# LOOCV, k-fold, sliding windows, split into 2 periods 

def combo_method_SKL_validation(X1,X2,Y, years, 
                                cv=LeaveOneOut(),
                                target='AR',
                                ):
    """
        Cross validate the bom combo method
        ARGS:
            DF: dataframe with nino,soi,region columns
            years: list of years to validate (or range or ndarray)
            cv: cross validator from sklearn with inherited "split" method
                default = LeaveOneOut()
    """
    if cv is None:
        cv=LeaveOneOut()

    i_sub = X1.index.year.isin(years)
    X1_sub, X2_sub, Y_sub = X1[i_sub], X2[i_sub], Y[i_sub]
    if not isinstance(years,np.ndarray):
        years = np.array(years)
    
    scores=[]
    i=0
    ## Iterate over dataset using cross validation arg
    for i_train, i_test in cv.split(X1_sub):
        # training and verification years
        years_tr = years[i_train]
        years_ver= years[i_test]
        
        # probability from combo method
        i_prob = combo_method_SKL(
            DF=DF_sub,
            training_range=years_tr,
            prediction_range=years_ver,
            show_scatter=False,plt=None,print_leps_skill=False)
        
        ## verification based on median of training data
        # use training data to set our median TR count
        obs_tr = DF_sub[target][i_train]
        obs_ver = DF_sub[target][i_test]
        median = obs_tr.median()
        # obs over verification space using that median
        y_ver = np.ones(len(years_ver),dtype=int)
        y_ver[obs_ver.values < median] = 1
        y_ver[obs_ver.values >= median] = 2
        # scores from probability vs observation
        scores.append(get_classification_scores(i_prob,y_ver))
        # add some meta data for easier analysis
        scores[i]['years_training']=years_tr
        scores[i]['years_verification']=years_ver
        #scores[i][]
        i=i+1
    # Calculate some averages
    # validation_means={"LEPS":0, }
    # for score in scores:
    #     mean_leps = mean_leps+score['LEPS skill']
    return scores

In [ ]:
# 9-12 그래프 다 만들기 (oni, roni, son, 1month lead, 3month lead, LEPS,BSS,HM for all training methods) -> 비교테이블 만들기 비교하기 쉽게. 
# Skill. Scroll to bottom to see skill over the past few years.

import analysis,prediction

#1990 was original. Suggest changing this to more recent, last 30 years.
# skill_start_year=1990
skill_start_year=1996
# skill_start_year=1994

tcs_yearly=tcs[tcs.index.month == 9]
#################### 
#### TSCV Loop + bss plot
####################


scores={}
plt.figure(figsize=[9,6])
for target,color in zip(['AR','AR-E','AR-N','AR-W','AR-NW','SPO','SPO-E','SPO-W'],['k','red','blue','green','magenta','cyan','brown','pink']):
# for target,color in zip(['SPO','SPO-E','SPO-W'],['k','magenta','cyan']):    
    scores[target]={'endyear':[],'LEPS skill':[],'BSS':[]}
    for endyear in range(skill_start_year,PREDICTION_YEAR):
        dtEnd = dt.datetime(endyear,9,1)
        Y = tcs_yearly[target].copy()
        X1 = nino_jas[:dtEnd].copy()
        X2 = soi_jas[:dtEnd].copy()
        pred_nino, pred_soi = prediction.combo_method_SKL(X1,X2,Y,prediction_year=endyear)

        Y_pred = (pred_nino+pred_soi)/2.0
        Y_obs = (Y[dtEnd] > Y[:dt.datetime(endyear,4,1)].median())+1
        score = analysis.get_classification_scores(Y_pred,[Y_obs])

        breakpoint()
        
        scores[target]['endyear'].append(endyear)
        scores[target]['LEPS skill'].append(float(score['LEPS skill'][0]))
        scores[target]['BSS'].append(score['BSS'])
        #print("DEBUG: %s %d LEPS skill:"%(target,endyear),score['LEPS skill'])
        #print("DEBUG: %s %d BSS skill:"%(target,endyear),score['BSS'])

 

    plt.plot(np.array(scores[target]['endyear']),np.array(scores[target]['BSS'],),
             label=target, color=color)
plt.title("Brier Skill Score")
plt.xlabel('test year')
plt.xlim([skill_start_year,PREDICTION_YEAR])
plt.ylim([-1,1])
plt.legend()
plt.show()

 

#######################
#### Rolling LEPS plot
#######################

 

plt.figure(figsize=[8,5])
for target,color in zip(['AR','AR-E','AR-N','AR-W','AR-NW',],['k','magenta','cyan','yellow','brown']):
# for target,color in zip(['SPO','SPO-E','SPO-W'],['k','magenta','cyan']):    
    # rolling mean
    dfi = pd.DataFrame(scores[target])
    # take rolling mean as new column (column index = 2, doesn't matter)
    dfi.insert(1,'rolling', dfi['LEPS skill'].rolling(window=5).mean(), allow_duplicates=True)
    plt.plot(np.array(dfi['endyear']),np.array(dfi['rolling'],),
             label=target, color=color)
plt.plot([skill_start_year+3,PREDICTION_YEAR],[20,20],'--k', alpha=0.3)
plt.plot([skill_start_year+3,PREDICTION_YEAR],[0,0],'--k', alpha=0.6)
plt.title("LEPS skill rolling 5 year mean", fontsize=20)
plt.xlabel('test year',fontsize=18)
plt.legend(fontsize=16)
#plt.savefig('figures/leps_rolling_mean.png')
plt.show()

 

#######################
#### Rolling BSS plot
#######################
plt.figure(figsize=[8,5])
for target,color in zip(['AR','AR-E','AR-N','AR-W','AR-NW',],['k','magenta','cyan','yellow','brown']):
# for target,color in zip(['SPO','SPO-E','SPO-W'],['k','magenta','cyan']):  
    # rolling mean
    dfi = pd.DataFrame(scores[target])
    # take rolling mean as new column (column index = 2, doesn't matter)
    dfi.insert(1,'rolling BSS', dfi['BSS'].rolling(window=5).mean(), allow_duplicates=True)
    plt.plot(np.array(dfi['endyear']),np.array(dfi['rolling BSS'],),
             label=target, color=color)
plt.plot([skill_start_year+4,PREDICTION_YEAR],[0,0],'--k', alpha=0.6)
plt.title("Brier skill score rolling 5 year mean",fontsize=20)
plt.xlabel('test year',fontsize=18)
plt.legend(fontsize=16)
#plt.savefig('figures/BSS_rolling_mean.png')
plt.show()

 

#######################
#### print last X years of scores
#######################
scores_df=None
scores_df_list=[]
for target in ['AR','AR-E','AR-N','AR-W','AR-NW','SPO','SPO-E','SPO-W']:
    dfi = pd.DataFrame(scores[target])
    dfi['target']=target
    # LEPS skill is already scalar; keep a shorter column name
    dfi['LEPS'] = dfi['LEPS skill']
    dfi.drop(columns=['LEPS skill'],inplace=True)
    scores_df_list.append(dfi)
    dfi.set_index(['target','endyear'],inplace=True)
    # print(dfi)
    if scores_df is None:
        scores_df = dfi
    else:
        scores_df = pd.concat([scores_df,dfi])


# Print the scores since 2020.
#print(scores_df[-3:])
#dfi[-3:]
scores_df.query('endyear >= 2020')